In [ ]:
import os
import gzip
import shutil
import urllib.request
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

In [5]:
class ASHD:
    def __init__(self, k=15, dim=64, iters=30, a=0.1, freq=3):
        self.k = k
        self.dim = dim
        self.iters = iters
        self.a = a
        self.freq = freq

    def fit_predict(self, H, signs, seed=42):
        n, m = H.shape
        np.random.seed(seed)
        
        X = np.random.normal(0, 0.01, (n, self.dim))
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        X = X / (norms + 1e-10)
        
        W = signs.copy()
        de = np.array(H.sum(axis=0)).flatten() + 1e-10
        De_inv = sparse.diags(1.0 / de)
        
        for t in range(self.iters):
            W_abs = np.abs(W)
            dv = np.array(H.dot(W_abs)).flatten() + 1e-10
            Dv_inv_sqrt = sparse.diags(1.0 / np.sqrt(dv))
            
            W_diag = sparse.diags(W)
            
            tmp = Dv_inv_sqrt.dot(X)
            tmp = H.T.dot(tmp)
            tmp = De_inv.dot(tmp)
            tmp = W_diag.dot(tmp)
            tmp = H.dot(tmp)
            X_new = Dv_inv_sqrt.dot(tmp)
            
            X = np.nan_to_num(X_new)
            norms = np.linalg.norm(X, axis=1, keepdims=True)
            X = X / (norms + 1e-10)
            
            if t > 0 and t % self.freq == 0:
                for j in range(m):
                    idx = H.getcol(j).indices
                    if len(idx) > 1:
                        sim = cosine_similarity(X[idx])
                        avg = (np.sum(sim) - len(idx)) / (len(idx) * (len(idx) - 1) + 1e-10)
                        W[j] = np.clip(W[j] + self.a * avg, -1.0, 1.0)
        
        km = KMeans(n_clusters=self.k, n_init=10, random_state=seed)
        labels = km.fit_predict(X)
        return labels, X

In [6]:
def get_data():
    url = "https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz"
    gz = "btc.csv.gz"
    csv = "soc-sign-bitcoinalpha.csv"
    
    if not os.path.exists(csv):
        opener = urllib.request.build_opener()
        opener.addheaders = [('User-agent', 'Mozilla/5.0')]
        urllib.request.install_opener(opener)
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as f_in, open(csv, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz)
    
    df = pd.read_csv(csv, names=['src', 'dst', 'rating', 'time'])
    nodes = np.unique(np.concatenate([df['src'], df['dst']]))
    node_map = {node: i for i, node in enumerate(nodes)}
    
    edges = []
    signs = []
    for src, g in df.groupby('src'):
        u = node_map[src]
        pos = [node_map[d] for d in g[g['rating'] > 0]['dst']]
        if pos:
            edges.append(list(set([u] + pos)))
            signs.append(1.0)
        neg = [node_map[d] for d in g[g['rating'] < 0]['dst']]
        if neg:
            edges.append(list(set([u] + neg)))
            signs.append(-1.0)
            
    rows = []
    cols = []
    for j, e in enumerate(edges):
        for i in e:
            rows.append(i)
            cols.append(j)
            
    H = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(nodes), len(edges)))
    return H, np.array(signs)

def calc_metrics(H, labels, X):
    sil = silhouette_score(X, labels, metric='cosine')
    
    clusters = np.unique(labels)
    bc_vals = []
    for c in clusters:
        idx = np.where(labels == c)[0]
        if len(idx) > 1:
            sim = cosine_similarity(X[idx])
            bc = (np.sum(sim) - len(idx)) / (len(idx) * (len(idx) - 1) + 1e-10)
            bc_vals.append(bc)
    bc = np.mean(bc_vals) if bc_vals else 0
    
    m = H.shape[1]
    vol_V = H.sum()
    mod = 0
    for c in clusters:
        idx = set(np.where(labels == c)[0])
        e_c = 0
        for j in range(m):
            e_nodes = set(H.getcol(j).indices)
            if e_nodes and e_nodes.issubset(idx):
                e_c += 1
        vol_c = H[list(idx), :].sum()
        mod += (e_c / m) - (vol_c / vol_V)**2
        
    return sil, bc, mod

if __name__ == "__main__":
    H, signs = get_data()
    
    model = ASHD(k=15, iters=30)
    labels, X = model.fit_predict(H, signs)
    
    sil, bc, mod = calc_metrics(H, labels, X)
    
    print(f"Results:")
    print(f"Silhouette: {sil:.4f}")
    print(f"Bio-Consistency: {bc:.4f}")
    print(f"Modularity: {mod:.4f}")

Results:
Silhouette: 0.8303
Bio-Consistency: 0.7596
Modularity: 0.1060
